In [ ]:
%cd ../../

In [ ]:
import sys
import datetime
from pathlib import Path
from typing import Any

import lightning as L
import torch
import yaml
import numpy as np
import polars as pl
import einops
from torch import Tensor, nn
from torch.nn import Module
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from lightning.pytorch.callbacks import RichProgressBar, LearningRateMonitor, ModelCheckpoint

from lightning.pytorch.loggers import TensorBoardLogger
from torch.nn import functional as F
from loguru import logger
from polars import DataFrame

In [ ]:
logger.remove()
logger.add(sys.stdout, level="INFO")

In [ ]:
STAGE = 2

In [ ]:
version = datetime.datetime.now().strftime("%m-%d_%H-%M-%S")
model_name = f"embedding-tuning_stage-{STAGE}"

with open('src/embedding_tuning/conf.yaml') as file:
    conf = yaml.safe_load(file)

if STAGE == 2:
    path = "weights/embedding_tuning/stage-1_02-27_10-38-36_epoch=9.ckpt"
    state_dict = torch.load(path)['state_dict']

# Prepare data for training

In [ ]:
map_mealtype2id = {
    "meat": 0,
    "fish": 1,
    "chicken": 2,
    "vegan": 3,
    "vegetarian": 4,
}

class DataEmbedding(Dataset):
    def __init__(self, conf: dict, stage: int) -> None:
        super().__init__()

        self.conf = conf
        self.stage = stage

        self.meals = self._load_meals()
        self.pairs = self._load_pairs()

    def _load_meals(self) -> DataFrame:
        meals = pl.read_parquet(self.conf['PATHS']['meals'])

        match self.stage:
            case 1:
                meals = (
                    meals
                    .select('meal_id', 'embedding', 'meal_type')
                    .unique()
                ) # fmt: skip
            case 2:
                meals = (
                    meals
                    .filter(pl.col('pcs') != -1)
                    .select(
                        'meal_id', 'embedding', 'meal_type', 'restaurant',
                        ((pl.col('pcs') - pl.min('pcs')) / (pl.max('pcs') - pl.min('pcs')) * 2.0).alias('pcs')
                    )
                    .unique()
                ) # fmt: skip
            case _:
                raise NotImplementedError()
            
        return meals

    def _load_pairs(self) -> DataFrame:
        match self.stage:
            case 1:
                pairs = self.meals.select('meal_id', 'meal_type')
                pairs = (
                    pairs
                    .join(pairs, how='cross')
                    .filter(
                        (1 == 1)
                        & (pl.col('meal_id') != pl.col('meal_id_right'))
                        & (pl.col('meal_type') == pl.col('meal_type_right'))
                    )
                    .select('meal_id', pl.col('meal_id_right').alias('meal_id_pos'))
                ) # fmt: skip
            case 2:
                pairs = self.meals.select('meal_id', 'meal_type', 'restaurant')
                pairs = (
                    pairs
                    .join(pairs, how='cross')
                    .filter(
                        (1 == 1)
                        & (pl.col('meal_id') != pl.col('meal_id_right'))
                        & (pl.col('meal_type') == pl.col('meal_type_right'))
                        & (pl.col('restaurant') == pl.col('restaurant_right'))
                    )
                    .select('meal_id', pl.col('meal_id_right').alias('meal_id_pos'), pl.col('restaurant'))
                ) # fmt: skip
            case _:
                raise NotImplementedError()

        return pairs


    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, index) -> dict:
        entry = self.pairs[index]

        meal_id, meal_id_pos = entry['meal_id'].item(), entry['meal_id_pos'].item()

        match self.stage:
            case 1:
                meal_info = self.meals.filter(pl.col('meal_id') == meal_id)
                meal_info_pos = self.meals.filter(pl.col('meal_id') == meal_id_pos)
            case 2:
                restaurant = entry['restaurant'].item()

                meal_info = self.meals.filter((pl.col('meal_id') == meal_id) & (pl.col('restaurant') == pl.lit(restaurant)))
                meal_info_pos = self.meals.filter((pl.col('meal_id') == meal_id_pos) & (pl.col('restaurant') == pl.lit(restaurant)))
            case _:
                raise NotImplementedError()


        embd_meal = meal_info['embedding'].item().to_numpy().copy().astype('float32')
        embd_meal_pos = meal_info_pos['embedding'].item().to_numpy().copy().astype('float32')

        if self.stage == 2:
            meal_pcs = meal_info['pcs'].item()
            meal_pcs_pos = meal_info_pos['pcs'].item()
        else:
            meal_pcs = 0.
            meal_pcs_pos = 0.


        meal_type = meal_info['meal_type'].item()
        meal_type_id = map_mealtype2id[meal_type]

        return {
            'meal': embd_meal,
            'meal_pos': embd_meal_pos,
            'meal_type': meal_type_id,
            'meal_pcs': np.array([meal_pcs, meal_pcs_pos]).astype('float32')
        }
    
# dataset = DataEmbedding(conf, STAGE)
# loader = DataLoader(dataset, conf['bsz'], shuffle=True)
# for batch in loader:
#     break

# batch['meal_pcs']

# Prepare model

In [ ]:
THRES_MIN = 1e-10

class ResNet(Module):
    def __init__(self, d_hid: int) -> None:
        super().__init__()

        self.ff = nn.Sequential(
            nn.Linear(d_hid, d_hid),
            nn.ReLU(),
            nn.Dropout(.3),
            nn.BatchNorm1d(d_hid),
        )

    def forward(self, X: Tensor) -> Tensor:
        return self.ff(X) + X


class EmbeddingLearner(Module):
    def __init__(
        self,
        d_embd: int,
        d_hid: int,
        closeness: float = 2.0,
    ) -> None:
        super().__init__()

        self.closeness = closeness      # This closeness isn't exact 1. to ensure the learned meal embeddings aren't exactly the same

        self.learner = nn.Sequential(
            ResNet(d_embd),
            nn.Linear(d_embd, d_hid),
        )

        self.lin_pos = ResNet(1)



    def trigger_train(
        self,
        meal: Tensor,
        meal_pos: Tensor,
        meal_type: Tensor,
        meal_pcs: Tensor,
        stage: int,
    ) -> Tensor:
        # meal, meal_pos: [bz, d_embd]
        # meal_type: [bz]

        # Encode
        meal = self.forward(meal)
        meal_pos = self.forward(meal_pos)
        # [bz, d_hid]

        
        # Calculate losses
        loss = self._calc_loss_meal_type(meal, meal_pos, meal_type)
        if stage == 2:
            loss = loss + self._calc_loss_pcs(meal, meal_pos, meal_pcs)

        return loss


    def _calc_loss_pcs(
        self,
        meal: Tensor,
        meal_pos: Tensor,
        meal_pcs: Tensor,
    ) -> Tensor:
        # meal, meal_pos: [bz, d_hid]
        # meal_pcs: [bz, 2]

        logger.debug("stage = 2, calculate loss for pcs")

        sim = torch.einsum("b d, b d -> b", [meal, meal_pos]).unsqueeze(-1)
        # [bz, 1]
        sim = 1.0 - sim     # Scale similarity from [-1, 1] to [0, 2]

        sim = self.lin_pos(sim).squeeze(-1)
        # [bz]

        pcs_diff = torch.abs(meal_pcs[:, 1] - meal_pcs[:, 0])

        loss = F.mse_loss(pcs_diff, sim)

        return loss

    def _calc_loss_meal_type(self, meal: Tensor, meal_pos: Tensor, meal_type: Tensor) -> Tensor:
        bz, d = meal_pos.shape

        # Apply: In-batch negative sampling: Duplicate embedding of positive meals
        meal_pos = einops.rearrange(
            einops.repeat(meal, "b d -> repeat b d", repeat=bz),
            "b x d -> b d x"
        )
        # [bz, d_hid, bz]

        logger.debug(f"After rearrange: meal_pos: {meal_pos.shape}")

        # Calculate similarity
        sim = torch.einsum("b d, b d y -> b y", [meal, meal_pos])
        # [bz, bz]
        sim = 1.0 - sim     # Scale similarity from [-1, 1] to [0, 2]

        # Create target tensor
        meal_types_others = einops.repeat(meal_type, "b -> repeat b", repeat=bz)
        meal_types_current = einops.repeat(meal_type, "b -> b repeat", repeat=bz)

        tgt = torch.full((bz, bz), THRES_MIN, device=meal_pos.device).masked_fill(meal_types_current == meal_types_others, self.closeness)
        # [bz, bz]


        # Calculate loss
        loss = F.mse_loss(sim, tgt)
        # [bz, bz]


        return loss


    def forward(self, meal: Tensor) -> Tensor:
        # meal: [bz, d_embd]

        # Encode
        meal = self.learner(meal)
        # [bz, d_hid]

        # Row-wise L2 normalize
        meal = meal / torch.norm(meal, dim=-1, p=2, keepdim=True)
        # [bz, d_hid]

        logger.debug(f"meal: {meal.shape}")

        return meal

# model = EmbeddingLearner(1024, 32)
# meal = torch.rand((conf['bsz'], 1024))
# meal_pos = torch.rand((conf['bsz'], 1024))
# meal_type = torch.randint(0, 5, (conf['bsz'],))
# meal_pcs = torch.rand((conf['bsz'], 2))

# loss = model.trigger_train(meal, meal_pos, meal_type, meal_pcs, stage=STAGE)
# sim.shape

In [ ]:
class LitEmbeddingLearner(L.LightningModule):
    def __init__(
        self,
        params: dict,
        lr: float = 1e-3,
        stage: int = 1
    ) -> None:
        super().__init__()
        self.save_hyperparameters()

        logger.info(f"stage = {stage}")

        self.model = EmbeddingLearner(**params)
        self.lr = lr
        self.stage = stage

    def forward(self, meal: Tensor) -> Any:
        return self.model(meal)

    def training_step(self, batch, batch_idx):
        loss = self.model.trigger_train(**batch, stage=self.stage)

        self.log("train_loss", loss, prog_bar=True, on_step=True, on_epoch=True)

        return loss

    def configure_optimizers(self):
        optimizer = AdamW(self.parameters(), lr=self.lr)

        scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1.0, end_factor=0.01, total_iters=conf['num_epoch'])

        return [optimizer], [{"scheduler": scheduler, "interval": "epoch"}]


# Train

In [ ]:
dataset = DataEmbedding(conf, STAGE)
loader = DataLoader(dataset, conf['bsz'], shuffle=True)

In [ ]:
params = {
    'd_hid': conf['d_hid'],
    'd_embd': conf['d_embd'],
}
litmodel = LitEmbeddingLearner(params, float(conf['lr']), stage=STAGE)
if STAGE == 2:
    litmodel.load_state_dict(state_dict)

In [ ]:
path_ckpt = Path(conf['PATHS']['ckpt'].replace("[date]", version))

trainer = L.Trainer(
    # devices=0,
    callbacks=[
        RichProgressBar(leave=True),
        LearningRateMonitor(logging_interval='step'),
        ModelCheckpoint(
            dirpath=path_ckpt.parent,
            filename=f"{path_ckpt.stem}_{{epoch}}",
            every_n_epochs=2
        )
    ],
    logger=TensorBoardLogger("tb_logs", name=model_name, version=version, default_hp_metric=False),
    # gradient_clip_val=1,
    max_epochs=conf['num_epoch'],
)

In [ ]:
trainer.fit(
    litmodel,
    loader,
    # ckpt_path='weights/embedding_tuning/stage-1_02-26_12-52-00_epoch=11.ckpt'
)

# Do inference

In [ ]:
if STAGE == 2:
    dataset = DataEmbedding(conf, 1)

with torch.no_grad():
    encoded = litmodel(torch.tensor(dataset.meals['embedding'].to_numpy(), dtype=torch.float32))

(
    dataset.meals
    .with_columns(pl.Series(encoded.numpy()).alias('embedding_encoded'))
    .write_parquet(f"data/processed/embedding_tuning/meal_embds_{version}.parquet")
)